In [ ]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_chroma import Chroma

from langchain_groq import ChatGroq

In [ ]:
GROQ_API_KEY = "secret_key"

In [7]:
PDF_FOLDER = Path("./pdfs")

documents = []

for pdf_file in PDF_FOLDER.glob("*.pdf"):

    print(f"Loading {pdf_file.name}")

    loader = PyPDFLoader(str(pdf_file))

    docs = loader.load()

    documents.extend(docs)

print(f"\nTotal Pages Loaded: {len(documents)}")

Loading Brain-Tumor-Dictionary_2014_web_3.12.21.pdf
Loading Brain-Tumor-Primer_PDF_ABTA1223.pdf
Loading Caregiver-Handbook_PDF_ABTA1223.pdf
Loading Newly-Diagnosed-Brochure_PDF_ABTA0325-2.pdf

Total Pages Loaded: 187


In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Total Chunks: {len(chunks)}")

Total Chunks: 730


In [9]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="../chroma_db"
)

print("Vector Database Created")

Vector Database Created


In [11]:
retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 6
    }
)

In [28]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [29]:
SYSTEM_PROMPT = """
You are a Brain Tumor Information Agent.

The input will ONLY be the name of a detected brain tumor.

Your task is to retrieve information from the provided medical context and generate a structured report.

Provide the following sections:

## Tumor Name

## Overview
A simple explanation of the tumor.

## Possible Causes
List common known causes or risk factors.

## Common Symptoms
List common symptoms associated with the tumor.

## Effects on the Brain and Body
Explain how the tumor may affect brain function, behavior, movement, vision, memory, or other body functions.

## Treatment Options
List common treatment approaches.

Rules:
- Use ONLY the provided context.
- Do not invent information.
- If information is unavailable, write:
  "Information not available in the medical database."
- Use clear patient-friendly language.
- Use bullet points wherever possible.
- Do not mention grades unless explicitly present in the retrieved context.
- Do not provide a diagnosis.
- Do not provide medical advice.

Context:
{context}

Detected Tumor:
{question}
"""

In [30]:
def brain_tumor_agent(question):

    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = SYSTEM_PROMPT.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    print("\n" + "="*80)
    print("ANSWER")
    print("="*80)

    print(response.content)

    print("\n")
    print("="*80)
    print("REFERENCES")
    print("="*80)

    for i, doc in enumerate(docs, start=1):

        source = doc.metadata.get(
            "source",
            "Unknown"
        )

        page = doc.metadata.get(
            "page",
            "Unknown"
        )

        print(
            f"{i}. {source} | Page {page}"
        )

In [31]:
brain_tumor_agent(
    "Glioma"
)


ANSWER
## Tumor Name
Glioma

## Overview
A glioma is a type of tumor that arises from the supportive tissue (called glial or neuroglial tissue) of the brain. It is a common primary brain tumor.

## Possible Causes
Information not available in the medical database.

## Common Symptoms
Common symptoms of gliomas can vary depending on the type and location of the tumor. However, some possible symptoms include:
* Neurological symptoms such as weakness, sensory changes, or balance difficulties
* Seizures
* Symptoms related to increased pressure in the brain, such as:
  * Nausea
  * Vomiting
  * Severe headaches (typically worse in the morning)
* Neurocognitive or memory issues

## Effects on the Brain and Body
Gliomas can affect brain function, behavior, movement, vision, memory, or other body functions, depending on the location and type of tumor. They can also cause increased pressure in the brain, leading to symptoms such as nausea, vomiting, and headaches.

## Treatment Options
Informa

In [32]:
brain_tumor_agent(
    "What are symptoms of Pituitary Tumor?"
)


ANSWER
## Tumor Name
Pituitary Tumor

## Overview
A pituitary tumor is a type of tumor that occurs in the pituitary gland, a small organ located at the base of the brain. The pituitary gland plays a crucial role in regulating various bodily functions, including hormone production.

## Possible Causes
Information not available in the medical database.

## Common Symptoms
Common symptoms of a pituitary tumor include:
* Hormone disturbances, which can cause:
	+ Water balance problems
	+ Abnormal growth
	+ Infertility
	+ Disruption of the menstrual cycle
	+ Hypertension
	+ Obesity
	+ Sleep disturbances
	+ Emotional changes
* Changes in appetite and desire for food
* Delayed or advanced sexual development
* Changes in sexual desire
* Obesity
* Delayed development
* Impaired vision
* Swollen optic nerve

## Effects on the Brain and Body
A pituitary tumor can affect the brain and body in several ways, including:
* Disrupting hormone production, leading to various bodily changes
* Causing pre